# 07. RNN / LSTM / GRU 体系

序列模型的核心问题是：当前输出不仅依赖当前输入，还依赖过去的信息。

## RNN / LSTM / GRU的形式化定义

            循环神经网络是一类通过隐状态递推处理序列数据的模型。其核心思想是使用历史隐藏状态作为当前时刻计算的一部分，从而实现对时间依赖或序列依赖的建模。

LSTM 与 GRU 是为缓解标准 RNN 长依赖建模困难而提出的门控循环结构。

## 输入、输出与参数化方式

            输入通常为序列张量 $X = (x_1, x_2, \dots, x_T)$。模型维护随时间变化的隐藏状态 $h_t$，并在需要时输出逐步预测或最终时刻预测。

LSTM 额外维护细胞状态 $c_t$；GRU 则通过更紧凑的门控机制简化状态更新。

## 结构分解与信息流

            标准 RNN 单元通过当前输入与上一时刻隐藏状态生成新的隐藏状态。LSTM 在此基础上引入输入门、遗忘门、输出门，从而对信息写入、保留与暴露进行细粒度控制；GRU 则使用更新门和重置门实现更简洁的门控递推。

这些门控机制并非附加修饰，而是解决长距离梯度传播困难的核心结构设计。

## 优化目标与训练机制

            训练通过时间反向传播进行。误差信号不仅沿网络深度传播，还会沿时间维度传播，因此优化难度通常高于前馈网络。

梯度消失和梯度爆炸是该类模型的典型问题，这也是门控结构、梯度裁剪与学习率控制的重要性所在。

## 计算复杂度、统计性质与工程代价

            RNN 家族在时间步上具有天然递归依赖，因此并行性弱于 Transformer。其单步状态更新开销较低，但长序列吞吐通常受限。

在中短序列预测、小规模时间序列和资源有限场景中，RNN / LSTM / GRU 仍然具有实际价值。

## 与相邻模型的差异

            与 MLP 相比，RNN 显式建模时间依赖。
与 Transformer 相比，RNN 的长程依赖建模和并行能力较弱，但状态递推结构更紧凑、在小模型场景下更经济。
与 SSM 相比，RNN 家族更传统，也更容易用门控视角理解序列记忆机制。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5)

xs = [1.5, 4.0, 6.5, 9.0]
for i, x in enumerate(xs, start=1):
    rect = plt.Rectangle((x - 0.6, 2.0), 1.2, 1.0, facecolor="#4C78A8", edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, 2.5, f"RNN Cell\nh{i}", ha="center", va="center", fontsize=11)
    ax.text(x, 1.3, f"x{i}", ha="center", fontsize=11)
    ax.annotate("", xy=(x, 2.0), xytext=(x, 1.55), arrowprops=dict(arrowstyle="->", lw=1.5))
for a, b in zip(xs[:-1], xs[1:]):
    ax.annotate("", xy=(b - 0.62, 2.5), xytext=(a + 0.62, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(10.8, 3.8, "隐藏状态沿时间递推", fontsize=12)
ax.set_title("RNN / LSTM / GRU 的时间展开视图")
plt.show()

## 先建立直觉

            序列问题和图像不同。图像是“同时出现”的空间结构，序列是“按时间展开”的结构。

例如一句话“我今天很开心”，你读到“开心”时，含义依赖前面“我今天很”。

RNN 家族的核心目标就是：

- 每处理一个时间步，就把当前信息和历史记忆融合起来
- 让模型带着“上下文状态”继续往后读

## 架构是怎么工作的

            普通 RNN 会维护一个隐藏状态 `h_t`。
每到新时间步，它都会用当前输入 `x_t` 和上一步隐藏状态 `h_{t-1}` 共同计算新的 `h_t`。

LSTM 和 GRU 的区别在于：它们发现“直接更新隐藏状态太粗暴”，于是引入门控机制：

- 哪些旧信息该忘
- 哪些新信息该写入
- 哪些信息该输出

所以门控结构本质上是在做“记忆管理”。

## 训练时到底发生了什么

            序列模型训练时会把误差沿时间反向传播，这叫做 BPTT（Backpropagation Through Time）。

问题也正出在这里：

- 序列太长时，梯度会越来越小，模型记不住远处信息
- 或者梯度会变得很大，训练不稳定

这就是为什么 LSTM / GRU 和梯度裁剪如此重要。

## 什么时候该用它

            RNN 家族适合：

- 中短序列时间序列预测
- 小规模序列建模任务
- 需要理解“状态递推”思想时

现代大模型里，RNN 已经不是主角，但它仍然是理解序列建模历史和基本问题的关键。

## 最常见的误区

            常见误区：

1. **误以为 LSTM 只是“更深的 RNN”**
   它的关键不是更深，而是门控记忆机制。

2. **误以为序列长就一定要用 RNN**
   长序列场景现在常常优先考虑 Transformer 或 SSM。

3. **误以为损失下降慢就是模型不行**
   序列模型往往训练更敏感，超参数和梯度稳定性非常关键。

## 1. RNN 的递推公式

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
time = np.linspace(0, 80, 1600)
series = np.sin(time) + 0.35 * np.sin(3 * time) + 0.10 * np.random.randn(len(time))

plt.figure(figsize=(14, 5))
plt.plot(time[:300], series[:300], color="steelblue")
plt.title("合成时间序列（局部）")
plt.show()

In [ ]:
def create_sequences(values, seq_len=24):
    X, y = [], []
    for i in range(len(values) - seq_len):
        X.append(values[i : i + seq_len])
        y.append(values[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

seq_len = 24
X, y = create_sequences(series, seq_len=seq_len)
X = X[:, :, None]

n_train = int(len(X) * 0.7)
n_val = int(len(X) * 0.15)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train : n_train + n_val], y[n_train : n_train + n_val]
X_test, y_test = X[n_train + n_val :], y[n_train + n_val :]

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=128, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=128, shuffle=False)

In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(self, cell_type="RNN", input_size=1, hidden_size=32):
        super().__init__()
        if cell_type == "RNN":
            self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        elif cell_type == "LSTM":
            self.rnn = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        elif cell_type == "GRU":
            self.rnn = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        else:
            raise ValueError(cell_type)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def train_sequence_model(model, train_loader, val_loader, epochs=35, lr=1e-3):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"val_loss": []}

    for _ in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        val_loss, val_total = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                pred = model(batch_x)
                loss = criterion(pred, batch_y)
                val_loss += loss.item() * batch_x.size(0)
                val_total += batch_x.size(0)
        history["val_loss"].append(val_loss / val_total)
    return history

In [ ]:
seq_models = {"RNN": SequenceRegressor("RNN"), "LSTM": SequenceRegressor("LSTM"), "GRU": SequenceRegressor("GRU")}
seq_histories = {}
trained_seq_models = {}

for name, model in seq_models.items():
    history = train_sequence_model(model, train_loader, val_loader, epochs=35, lr=1e-3)
    seq_histories[name] = history
    trained_seq_models[name] = model
    print(name, "最终验证损失:", round(history["val_loss"][-1], 6))

In [ ]:
plt.figure(figsize=(12, 6))
for name, history in seq_histories.items():
    plt.plot(history["val_loss"], label=name)
plt.title("不同序列模型的验证损失曲线")
plt.legend()
plt.show()

In [ ]:
def predict_series(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            preds.append(model(batch_x).numpy())
            targets.append(batch_y.numpy())
    return np.concatenate(preds), np.concatenate(targets)

test_records = []
test_predictions = {}
targets = None
for name, model in trained_seq_models.items():
    preds, current_targets = predict_series(model, test_loader)
    mse = np.mean((preds - current_targets) ** 2)
    test_records.append({"模型": name, "Test MSE": mse})
    test_predictions[name] = preds
    targets = current_targets

test_df = pd.DataFrame(test_records).sort_values("Test MSE")
test_df

In [ ]:
best_name = test_df.iloc[0]["模型"]
best_preds = test_predictions[best_name]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=test_df, x="模型", y="Test MSE", ax=axes[0], palette="mako")
axes[0].set_title("RNN / LSTM / GRU 的测试误差对比")

axes[1].plot(targets[:200], label="真实值", color="black")
axes[1].plot(best_preds[:200], label=f"{best_name} 预测值", color="crimson", alpha=0.8)
axes[1].set_title(f"{best_name} 在测试集上的预测效果")
axes[1].legend()

plt.tight_layout()
plt.show()